In [1]:
import pandas as pd
import glob
import os

# =========================
# CONFIG (Convolutional FMNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV":    "prune_layers_CONV",
    "FHL":     "prune_layers_FHL",
    "SHL":     "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL":     "prune_layers_ALL"
}

BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"
FILE_PATTERN = "convol_{:.1f}_{}_run_*.txt"

BATCH_SIZES = [64, 1024]
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]

# =========================
# MAIN AVERAGING LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    for bs in BATCH_SIZES:
        for p in PRUNING_LEVELS:

            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, bs)))

            if not files:
                print(f"[WARNING] No files for {prune_layer}, p={p}, bs={bs}")
                continue

            # =========================
            # LOAD ALL RUNS
            # =========================
            dfs = []
            for f in files:
                df = pd.read_csv(f, sep=r"\s+")
                df.columns = df.columns.str.strip()

                df["CE_Train"]    = pd.to_numeric(df["CE_Train"],    errors="coerce")
                df["CE_TEST"]     = pd.to_numeric(df["CE_TEST"],     errors="coerce")
                df["Accuracy(%)"] = pd.to_numeric(df["Accuracy(%)"], errors="coerce")

                dfs.append(df)

            all_runs = pd.concat(dfs, ignore_index=True)

            # =========================
            # GROUP + AVERAGE
            # =========================
            avg_df = all_runs.groupby("Batch_Number", as_index=False).agg(
                Avg_CE_Train=("CE_Train",    "mean"),
                Avg_CE_Test= ("CE_TEST",     "mean"),
                Avg_Accuracy=("Accuracy(%)", "mean"),
                Num_Runs=    ("CE_TEST",     "count")
            )

            # =========================
            # SAVE CSV
            # =========================
            out_csv = os.path.join(
                folder,
                f"averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv"
            )

            avg_df.to_csv(out_csv, index=False)
            print(f"[SAVED] {out_csv}")


[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST\prune_layers_CONV\p-percentage_0.0\batch_size_64\averaged_runs_conv_CONV_p_0.0_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST\prune_layers_CONV\p-percentage_0.1\batch_size_64\averaged_runs_conv_CONV_p_0.1_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST\prune_layers_CONV\p-percentage_0.2\batch_size_64\averaged_runs_conv_CONV_p_0.2_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST\prune_layers_CONV\p-percentage_0.3\batch_size_64\averaged_runs_conv_CONV_p_0.3_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST\prune_layers_CONV\p-percentage_0.4\batch_size_64\averaged_runs_conv_CONV_p_0.4_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Ne

In [2]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV":    "prune_layers_CONV",
    "FHL":     "prune_layers_FHL",
    "SHL":     "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL":     "prune_layers_ALL"
}

BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"
AVG_FILE_PATTERN = "averaged_runs_conv_*_p_{}_bs_{}.csv"

BATCH_SIZES = [64, 1024]
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]

LN10 = np.log(10)

# =========================
# COLOR LIST (Custom)
# =========================
COLOR_LIST = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#800080",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#B9D9EB", "#17becf"
]

# inverse is required so P%=0 -> teal (top line for CE), P%=100 -> blue (bottom)
COLOR_LIST = COLOR_LIST[::-1]

# =========================
# OUTPUT DIRECTORY
# =========================
PLOT_ROOT = os.path.join(BASE_DIR_ROOT, "Avg_Plots")
os.makedirs(PLOT_ROOT, exist_ok=True)

# =========================
# STYLE (Nature-like)
# =========================
plt.rcParams.update({
    "font.size":       18,
    "axes.titlesize":  18,
    "axes.labelsize":  18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 15
})

# =========================
# MAIN PLOTTING LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])
    plot_dir = os.path.join(PLOT_ROOT, prune_layer)
    os.makedirs(plot_dir, exist_ok=True)

    for bs in BATCH_SIZES:

        avg_dfs = {}

        # =========================
        # LOAD ALL AVERAGED FILES
        # =========================
        for p in PRUNING_LEVELS:
            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            files  = glob.glob(os.path.join(folder, AVG_FILE_PATTERN.format(p, bs)))

            if not files:
                continue

            avg_dfs[p] = pd.read_csv(files[0])

        if not avg_dfs:
            continue

        sorted_items = sorted(avg_dfs.items())

        # =========================
        # CE TRAIN
        # =========================
        plt.figure(figsize=(12, 6))

        for idx, (p, df) in enumerate(sorted_items):
            color = COLOR_LIST[idx % len(COLOR_LIST)]
            plt.plot(df["Batch_Number"], df["Avg_CE_Train"],
                     label=f"P%={int(p * 100)}",
                     color=color)

        plt.xlabel("Batch Number")
        plt.ylabel("Average CE")
        plt.ylim(0, 2.7)

        plt.text(0.06, LN10, r"$\ln(10)$",
                 transform=plt.gca().get_yaxis_transform(),
                 fontsize=14, va="center")

        plt.text(0.5, 0.95,
                 f"Convolutional FMNIST ({prune_layer}-layer(s))",
                 transform=plt.gca().transAxes,
                 ha="center", va="top")
        plt.text(0.5, 0.90,
                 f"(Training-Vectors, BS={bs})",
                 transform=plt.gca().transAxes,
                 ha="center", va="top", fontsize=16)

        handles, labels = plt.gca().get_legend_handles_labels()
        plt.legend(handles[::-1], labels[::-1], frameon=False)

        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            os.path.join(plot_dir, f"Conv_Avg_CE_Train_{prune_layer}_BS{bs}.png"),
            dpi=300, bbox_inches="tight"
        )
        plt.close()

        # =========================
        # CE TEST
        # =========================
        plt.figure(figsize=(12, 6))

        for idx, (p, df) in enumerate(sorted_items):
            color = COLOR_LIST[idx % len(COLOR_LIST)]
            plt.plot(df["Batch_Number"], df["Avg_CE_Test"],
                     label=f"P%={int(p * 100)}",
                     color=color)

        plt.xlabel("Batch Number")
        plt.ylabel("Average CE")
        plt.ylim(0, 2.7)

        plt.text(0.06, LN10, r"$\ln(10)$",
                 transform=plt.gca().get_yaxis_transform(),
                 fontsize=14, va="center")

        plt.text(0.5, 0.95,
                 f"Convolutional FMNIST ({prune_layer}-layer(s))",
                 transform=plt.gca().transAxes,
                 ha="center", va="top")
        plt.text(0.5, 0.90,
                 f"(Test-Vectors, BS={bs})",
                 transform=plt.gca().transAxes,
                 ha="center", va="top", fontsize=16)

        handles, labels = plt.gca().get_legend_handles_labels()
        plt.legend(handles[::-1], labels[::-1], frameon=False)

        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            os.path.join(plot_dir, f"Conv_Avg_CE_Test_{prune_layer}_BS{bs}.png"),
            dpi=300, bbox_inches="tight"
        )
        plt.close()

        # =========================
        # ACCURACY
        # =========================
        plt.figure(figsize=(12, 6))

        for idx, (p, df) in enumerate(sorted_items):
            color = COLOR_LIST[idx % len(COLOR_LIST)]
            plt.plot(df["Batch_Number"], df["Avg_Accuracy"],
                     label=f"P%={int(p * 100)}",
                     color=color)

        plt.xlabel("Batch Number")
        plt.ylabel("Average Accuracy (%)")
        plt.ylim(0, 100)

        plt.text(0.5, 0.55,
                 f"Convolutional FMNIST ({prune_layer}-layer(s))",
                 transform=plt.gca().transAxes,
                 ha="center", va="top")
        plt.text(0.5, 0.50,
                 f"(Test-Vectors, BS={bs})",
                 transform=plt.gca().transAxes,
                 ha="center", va="top", fontsize=16)

        plt.legend(frameon=False)

        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            os.path.join(plot_dir, f"Conv_Avg_Accuracy_{prune_layer}_BS{bs}.png"),
            dpi=300, bbox_inches="tight"
        )
        plt.close()

        print(f"[DONE] {prune_layer}, BS={bs}")


[DONE] CONV, BS=64
[DONE] CONV, BS=1024
[DONE] FHL, BS=64
[DONE] FHL, BS=1024
[DONE] SHL, BS=64
[DONE] SHL, BS=1024
[DONE] FHL+SHL, BS=64
[DONE] FHL+SHL, BS=1024
[DONE] ALL, BS=64
[DONE] ALL, BS=1024


## New version of the code

In [ ]:
"""
This script loads averaged training results for a Convolutional FMNIST model
under different pruning-layer configurations and batch sizes, and generates
compact publication-quality plots.

For each pruning configuration (CONV, FHL, SHL, FHL+SHL, ALL)
and each batch size (64, 1024), the script:

1. Loads averaged CSV files for pruning levels (0%-100%).
2. Extracts:
      - Average Cross-Entropy (Train)
      - Average Cross-Entropy (Test)
      - Average Accuracy
3. Generates three plots per configuration:
      - CE Train vs Batch Number
      - CE Test vs Batch Number
      - Accuracy vs Batch Number
4. Places titles and legends inside the plotting area to save space.
5. Displays plots in the notebook.
6. Saves all figures at 300 DPI in a structured output directory.
"""

import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV":    "prune_layers_CONV",
    "FHL":     "prune_layers_FHL",
    "SHL":     "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL":     "prune_layers_ALL"
}

BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"
AVG_FILE_PATTERN = "averaged_runs_conv_*_p_{}_bs_{}.csv"

BATCH_SIZES = [64, 1024]
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]

LN10 = np.log(10)

# =========================
# COLOR LIST
# =========================
COLOR_LIST = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#800080",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#B9D9EB", "#17becf"
][::-1]

# =========================
# OUTPUT DIRECTORY
# =========================
PLOT_ROOT = os.path.join(BASE_DIR_ROOT, "Avg_Plots")
os.makedirs(PLOT_ROOT, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size":       18,
    "axes.titlesize":  18,
    "axes.labelsize":  18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14
})

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])
    plot_dir = os.path.join(PLOT_ROOT, prune_layer)
    os.makedirs(plot_dir, exist_ok=True)

    for bs in BATCH_SIZES:

        avg_dfs = {}

        # -------------------------
        # LOAD DATA
        # -------------------------
        for p in PRUNING_LEVELS:
            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            files  = glob.glob(os.path.join(folder, AVG_FILE_PATTERN.format(p, bs)))

            if files:
                avg_dfs[p] = pd.read_csv(files[0])

        if not avg_dfs:
            continue

        sorted_items = sorted(avg_dfs.items())

        # ======================================================
        # HELPER: build a single figure and save it
        # ======================================================
        def make_plot(y_col, ylabel, ylim, subtitle, filename,
                      reverse_legend, legend_loc, show_ln10):

            fig, ax = plt.subplots(figsize=(12, 6))

            for idx, (p, df) in enumerate(sorted_items):
                color = COLOR_LIST[idx % len(COLOR_LIST)]
                ax.plot(
                    df["Batch_Number"], df[y_col],
                    label=f"P%={int(p * 100)}",
                    color=color
                )

            ax.set_xlabel("Batch Number")
            ax.set_ylabel(ylabel)
            ax.set_ylim(*ylim)

            if show_ln10:
                ax.text(0.06, LN10, r"$\ln(10)$",
                        transform=ax.get_yaxis_transform(),
                        fontsize=14, va="center")
                title_y, sub_y = 0.95, 0.90
            else:
                title_y, sub_y = 0.55, 0.50

            ax.text(0.5, title_y,
                    f"Convolutional FMNIST ({prune_layer}-layer(s))",
                    transform=ax.transAxes, ha="center", va="top")
            ax.text(0.5, sub_y, f"{subtitle}, BS={bs}",
                    transform=ax.transAxes, ha="center", va="top", fontsize=16)

            handles, labels = ax.get_legend_handles_labels()
            if reverse_legend:
                ax.legend(handles[::-1], labels[::-1],
                          frameon=False, loc=legend_loc)
            else:
                ax.legend(frameon=False, loc=legend_loc)

            ax.grid(True)
            plt.tight_layout()
            fig.savefig(os.path.join(plot_dir, filename),
                        dpi=300, bbox_inches="tight")
            plt.show()
            plt.close(fig)

        # -------------------------
        # CE TRAIN
        # -------------------------
        make_plot(
            y_col="Avg_CE_Train",
            ylabel="Average CE",
            ylim=(0, 2.7),
            subtitle="Training-Vectors",
            filename=f"Conv_Avg_CE_Train_{prune_layer}_BS{bs}.png",
            reverse_legend=True,
            legend_loc="upper right",
            show_ln10=True
        )

        # -------------------------
        # CE TEST
        # -------------------------
        make_plot(
            y_col="Avg_CE_Test",
            ylabel="Average CE",
            ylim=(0, 2.7),
            subtitle="Test-Vectors",
            filename=f"Conv_Avg_CE_Test_{prune_layer}_BS{bs}.png",
            reverse_legend=True,
            legend_loc="upper right",
            show_ln10=True
        )

        # -------------------------
        # ACCURACY
        # -------------------------
        make_plot(
            y_col="Avg_Accuracy",
            ylabel="Average Accuracy (%)",
            ylim=(0, 100),
            subtitle="Test-Vectors",
            filename=f"Conv_Avg_Accuracy_{prune_layer}_BS{bs}.png",
            reverse_legend=False,
            legend_loc="lower right",
            show_ln10=False
        )

        print(f"[DONE] {prune_layer}, BS={bs}")
